# 🧬 MTase Topology Analyzer

**Automatic analysis and classification of DNA methyltransferases (MTases) based on beta-sheet topology**

---
### 📋 Features:
* Analyze structures by PDB ID, AlphaFold ID, or uploaded file
* Determine beta-strand order and directions (↑/↓)
* Classify into classes A, B, C, D, E, F
* Visualize linear, 2D, and 3D topology

---

In [0]:
# @title ⚡ Installation (Run First)

import os
import sys
import time

print("🚀 Installing MTase Topology Analyzer...")
print("=" * 60)

# Clone repository
if not os.path.exists("MTase_topology_analyser"):
    !git clone https://github.com/MVolobueva/MTase_topology_analyser.git
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

# Change directory
os.chdir("MTase_topology_analyser")

# Install dependencies
print("\n📦 Installing Python dependencies...")
!pip install -q streamlit pandas numpy scipy biopython plotly py3dmol

print("\n" + "=" * 60)
print("✅ Installation complete!")
print("=" * 60)

In [0]:
# @title 🎛️ Choose Analysis Mode

mode = "Web App"  # @param ["Web App", "Quick Notebook Analysis"]

if mode == "Web App":
    print("🚀 Launching Streamlit Web App...")
    print("📌 After startup, click the link below")
    print("\n" + "=" * 60)
    
    !pkill -f streamlit 2>/dev/null
    
    !streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats false &
    
    time.sleep(5)
    
    from google.colab import output
    output.serve_kernel_port_as_window(8501)
    
    from IPython.display import HTML, display
    display(HTML('<div style="background-color: #d4edda; padding: 15px; border-radius: 10px; margin: 10px 0;">'
        '<h3>✅ Streamlit App is Running!</h3>'
        '<p>Click the link below to open the MTase Topology Analyzer:</p>'
        '<a href="https://localhost:8501" target="_blank" style="font-size: 18px;">🔗 Open MTase Topology Analyzer</a>'
        '<p><small>If the link doesn\'t work, go to the <b>Ports</b> tab and click the 🌐 icon next to port 8501.</small></p>'
        '</div>'))
else:
    print("🔬 Quick Notebook Analysis Mode")
    print("=" * 60)
    
    import pandas as pd
    import io
    import re
    from IPython.display import display, Markdown
    
    from analyzer import MTaseAnalyzer
    from utils.helpers import download_structure
    from classifier import classify_topology
    
    def analyze_single_structure(id_value, source_type):
        print(f"\n🔬 Analyzing: {id_value} ({source_type})")
        print("-" * 50)
        
        try:
            if source_type == 'pdb':
                result_files = download_structure(id_value, source='pdb')
            elif source_type == 'alphafold':
                result_files = download_structure(id_value, source='alphafold')
            else:
                print("   Unsupported source type")
                return None
            
            if not result_files:
                print(f"   ❌ Failed to load {id_value}")
                return None
            
            analyzer = MTaseAnalyzer()
            if not analyzer.load_dssp(result_files['dssp']):
                print(f"   ❌ Failed to load DSSP")
                return None
            
            analyzer.find_all_strands()
            analyzer.build_sheet_adjacency()
            
            motifs = analyzer.find_all_motifs()
            motifs = analyzer.filter_motifs_by_topology(motifs)
            
            if not motifs:
                print(f"   ⚠️ No motifs found")
                return None
            
            results = []
            for motif_data in motifs:
                chain = motif_data['chain']
                motif_text = motif_data['text']
                motif_res = motif_data['res']
                motif_position = f"{motif_res}-{motif_res + len(motif_text) - 1}"
                
                result = analyzer.analyze_topology(motif_data=motif_data)
                if not result:
                    continue
                
                old_stdout = sys.stdout
                new_stdout = io.StringIO()
                sys.stdout = new_stdout
                analyzer.print_linear_topology_from_result(result)
                sys.stdout = old_stdout
                
                topology_str = new_stdout.getvalue()
                lines = topology_str.strip().split('\n')
                topology_line = ""
                for line in lines:
                    if ('↑' in line or '↓' in line) and '[' in line and ']' in line:
                        topology_line = line.strip()
                        break
                
                if not topology_line:
                    continue
                
                classification = classify_topology(topology_line, motif_text)
                
                results.append({
                    'Chain': chain,
                    'Motif': motif_text,
                    'Position': motif_position,
                    'Class': classification['class'],
                    'Confidence': classification['confidence'],
                    'Strand Order': classification['strand_order'],
                    'Has S0': classification['has_s0'],
                    'Has S-1': classification['has_s-1'],
                    'Gap S6-S7': classification['gap_s6_s7']
                })
            
            if results:
                df = pd.DataFrame(results)
                display(Markdown(f"### 📊 Results for {id_value}"))
                display(df)
                return results
            else:
                print(f"   ⚠️ No valid topology found")
                return None
                
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
            return None
    
    source_id = "3S1S"
    source_type = "pdb"
    
    analyze_single_structure(source_id, source_type)

## 📚 Help & Documentation

### How to Use

**Option 1: Interactive Web App (Recommended)**
1. Select "Web App" mode above
2. Click the generated link
3. Enter PDB ID (e.g., `3S1S`) or AlphaFold ID
4. Click "Run Analysis"

**Option 2: Quick Notebook Analysis**
1. Select "Quick Notebook Analysis" mode
2. Enter a PDB ID or AlphaFold ID
3. Results appear directly in the notebook

### Links
- [GitHub Repository](https://github.com/MVolobueva/MTase_topology_analyser)

### Contact
M. Samokhina – mashila6799@gmail.com